In [ ]:
from pythtb import Lattice, Mesh, TBModel, WFArray
import numpy as np

In [ ]:
lattice = Lattice(
    lat_vecs=[[1.0]], orb_vecs=[[0.0], [1 / 3], [2 / 3]], periodic_dirs=[0]
)

In [ ]:
mesh = Mesh(dim_k=1, axis_types=["k"])

In [ ]:
mesh.build_grid([20])

In [ ]:
wfa = WFArray(lattice=lattice, mesh=mesh)

In [ ]:
def set_model(t: float, delta: float, lmbda: float) -> TBModel:
    """Periodic three-site model at parameter lmbda."""

    model = TBModel(lattice=lattice)

    # nearest-neighbour hoppings (last hop wraps to the next cell)
    model.set_hop(t, 0, 1, [0])
    model.set_hop(t, 1, 2, [0])
    model.set_hop(t, 2, 0, [1])

    onsite = [delta * -np.cos(2 * np.pi * (lmbda - idx / 3)) for idx in range(3)]
    model.set_onsite(onsite)
    return model


model = set_model(t=1.0, delta=1.0, lmbda=0.0)
wfa.solve_model(model)

In [ ]:
wfa[2]

In [ ]:
t = -1.3
delta = 2.0

model = TBModel(lattice=lattice)

# nearest-neighbour hoppings (last hop wraps to the next cell)
model.set_hop(t, 0, 1, [0])
model.set_hop(t, 1, 2, [0])
model.set_hop(t, 2, 0, [1])

# Per-orbital onsite as lambdas of lmbda
onsite = [
    lambda lmbda: delta * -np.cos(2 * np.pi * (lmbda - 0 / 3)),
    lambda lmbda: delta * -np.cos(2 * np.pi * (lmbda - 1 / 3)),
    lambda lmbda: delta * -np.cos(2 * np.pi * (lmbda - 2 / 3)),
]

model.set_onsite(onsite)
print(model)

In [ ]:
mesh = Mesh(
    dim_k=1,
    dim_lambda=1,
    axis_types=["k", "l"],  # first axis: crystal momentum; second: adiabatic parameter
    axis_names=["kx", "lmbda"],
)

In [ ]:
mesh.build_grid(shape=(31, 21), gamma_centered=True, lambda_start=0.0, lambda_stop=1.0)

In [ ]:
mesh.loop_axis(axis_idx=1, component_idx=1)  # form the lambda axis into a loop

In [ ]:
mesh.close_axis(
    axis_idx=1, component_idx=1
)  # indicate that the end of the loop completes the cycle (endpoint included)
print(mesh)

In [ ]:
wfa = WFArray(lattice, mesh)

In [ ]:
kpts = np.linspace(-0.5, 0.5, 21)[:, None]
H = model.hamiltonian(kpts, lmbda=np.linspace(0, 1, 10, endpoint=True))
H.shape

In [ ]:
wfa.solve_model(model=model)